In [ ]:
import requests
#  CONFIGURATION

HF_API_KEY = "YOUR_API_KEY"
MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
API_URL = f"https://api-inference.huggingface.co/models/{MODEL}"

HEADERS = {
    "Authorization": f"Bearer {HF_API_KEY}",
    "Content-Type": "application/json"
}

In [ ]:

#  SYSTEM PROMPT (Prompt Engineering)

SYSTEM_PROMPT = """You are a friendly and caring health information assistant.
Your job is to help people understand general health topics clearly and simply.

Rules you must always follow:
1. Be warm, supportive, and easy to understand. Avoid complex medical jargon.
2. Provide GENERAL health education only — never diagnose or prescribe.
3. Always remind the user to consult a doctor for personal medical decisions.
4. Keep responses concise (under 200 words).
5. If a question is about a medication, explain what it is generally used for
   but always recommend consulting a doctor or pharmacist for dosage.
6. Never recommend stopping a prescribed medication.
"""

In [ ]:
#  SAFETY FILTER

EMERGENCY_KEYWORDS = [
    "chest pain", "heart attack", "can't breathe", "difficulty breathing",
    "stroke", "unconscious", "overdose", "severe bleeding", "seizure",
    "anaphylaxis", "allergic reaction", "suicide", "self-harm", "emergency"
]

EMERGENCY_RESPONSE = (
    "\n EMERGENCY ALERT\n"
    "Your symptoms may require immediate medical attention.\n"
    "Please call emergency services (e.g. 115 in Pakistan, 911 in the US)\n"
    "or go to the nearest emergency room RIGHT NOW.\n"
    "Do not wait or rely on online advice in an emergency.\n"
)

def is_emergency(user_input: str) -> bool:
    """Check if the query contains emergency keywords."""
    text = user_input.lower()
    return any(keyword in text for keyword in EMERGENCY_KEYWORDS)

In [ ]:
#  API CALL

def build_prompt(conversation_history: list, new_question: str) -> str:
    """
    Build a Mistral-style chat prompt using conversation history.
    Mistral Instruct uses [INST] ... [/INST] tags.
    """
    prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\nUser question: {new_question} [/INST]"

    # Add recent conversation turns for context (last 3 turns)
    if conversation_history:
        history_text = "\n".join(
            f"User: {turn['user']}\nAssistant: {turn['bot']}"
            for turn in conversation_history[-3:]
        )
        prompt = (
            f"<s>[INST] {SYSTEM_PROMPT}\n\n"
            f"Previous conversation:\n{history_text}\n\n"
            f"User question: {new_question} [/INST]"
        )
    return prompt


def ask_health_question(question: str, conversation_history: list) -> str:
    """Send a health question to the Hugging Face model and return the answer."""

    # Safety filter — check before calling the API
    if is_emergency(question):
        return EMERGENCY_RESPONSE

    prompt = build_prompt(conversation_history, question)

    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 300,      # Limit response length
            "temperature": 0.7,          # Balanced creativity
            "top_p": 0.9,
            "do_sample": True,
            "return_full_text": False    # Return only the new reply
        }
    }

    try:
        response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30)
        response.raise_for_status()
        result = response.json()

        # Extract the generated text
        if isinstance(result, list) and len(result) > 0:
            answer = result[0].get("generated_text", "").strip()
        else:
            answer = "Sorry, I could not get a response. Please try again."

        # Append disclaimer
        answer += (
            "\n\n📋 Reminder: This is general health information only. "
            "Please consult a qualified healthcare professional for personal medical advice."
        )
        return answer

    except requests.exceptions.Timeout:
        return "The request timed out. The model may be loading — please try again in a moment."
    except requests.exceptions.HTTPError as e:
        return f"API error: {e}. Check your API key and try again."
    except Exception as e:
        return f"Unexpected error: {e}"

#  MAIN CHAT LOOP

def main():
    print("=" * 60)
    print("   General Health Query Chatbot")
    print("  Powered by Mistral-7B via Hugging Face")
    print("  Type 'quit' or 'exit' to stop")
    print("=" * 60)
    print()

    conversation_history = []

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ("quit", "exit", "bye"):
            print("\nChatbot: Take care and stay healthy! Goodbye.")
            break

        print("\nChatbot: Thinking...\n")
        response = ask_health_question(user_input, conversation_history)
        print(f"Chatbot: {response}\n")
        print("-" * 60)

        # Save turn to conversation history
        conversation_history.append({
            "user": user_input,
            "bot": response
        })


if __name__ == "__main__":
    main()